<a href="https://colab.research.google.com/github/kuroshkarimi/Machine-Learning-and-Data-Analysis/blob/main/Association%20Rule/Market_Basket_Optimisation_apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Product Co-occurrence Analysis with Apriori

1. Business scenario

A supermarket has collected transaction records from its customers. Each transaction contains the products purchased together in one shopping basket.

This data could be found on this repository:
https://github.com/kuroshkarimi/Machine-Learning-and-Data-Analysis/raw/refs/heads/main/Association%20Rule/Market_Basket_Optimisation.csv

The supermarket wants to understand which products are frequently purchased together so that it can improve:

Product placement
Cross-selling recommendations
Promotional campaigns
Product bundles
“Frequently bought together” suggestions

2. Project objective

Use the Apriori association-rule algorithm to discover product combinations and association rules from customer transactions. Consider the popularity of over 0.001, reliability of over 0.1 and strength of over 1.01 for the exploratory analysis.


3. Main data-science task

Given a transaction dataset containing the products purchased by customers, identify:

- Individual products occurring in at least 0.1% of transactions.
- Product pairs purchased together.
- Product combinations containing two to four products.

4. Dataset structure

The raw dataset may have one transaction per row:

shrimp, almonds, avocado, vegetables mix
burgers, meatballs, eggs
chutney
turkey, avocado
mineral water, milk, energy bar

5. Questions to answer

Considering a strong rule as a rule with the following characteristics, answer the following questions:

(min_support = 0.004,
min_confidence = 0.2,
min_lift = 4)


- Most frequent individual products
- Most popular itemsets
- The strong rules satisfying the complete project thresholds
- Highest-confidence strong rules
- Highest-lift strong rules



## Importing the libraries

In [23]:
!pip install apyori

In [24]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from apyori import apriori
from collections import Counter


## Data Preprocessing

In [25]:
# Loading dataset
url = "https://github.com/kuroshkarimi/Machine-Learning-and-Data-Analysis/raw/refs/heads/main/Association%20Rule/Market_Basket_Optimisation.csv"
dataset = pd.read_csv(url, header = None)
dataset

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
1,burgers,meatballs,eggs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chutney,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,turkey,avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mineral water,milk,energy bar,whole wheat rice,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7496,butter,light mayo,fresh bread,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7497,burgers,frozen vegetables,eggs,french fries,magazines,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7498,chicken,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7499,escalope,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
# Checking the general charecteristics of the data

dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7501 entries, 0 to 7500
Data columns (total 20 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       7501 non-null   object
 1   1       5747 non-null   object
 2   2       4389 non-null   object
 3   3       3345 non-null   object
 4   4       2529 non-null   object
 5   5       1864 non-null   object
 6   6       1369 non-null   object
 7   7       981 non-null    object
 8   8       654 non-null    object
 9   9       395 non-null    object
 10  10      256 non-null    object
 11  11      154 non-null    object
 12  12      87 non-null     object
 13  13      47 non-null     object
 14  14      25 non-null     object
 15  15      8 non-null      object
 16  16      4 non-null      object
 17  17      4 non-null      object
 18  18      3 non-null      object
 19  19      1 non-null      object
dtypes: object(20)
memory usage: 1.1+ MB


In [27]:

# ------------------------------------------------------------
# 1. Prepare transactions

transactions = []

for _, row in dataset.iterrows():
    transaction = {
        str(item).strip()
        for item in row.dropna()
        if str(item).strip()
    }
    transactions.append(sorted(transaction))

n_transactions = len(transactions)


In [28]:
# ------------------------------------------------------------
# 2. Individual-item support

item_counter = Counter(
    item
    for transaction in transactions
    for item in transaction
)

individual_items = pd.DataFrame({
    "item": item_counter.keys(),
    "transaction_count": item_counter.values()
})

individual_items["support"] = (
    individual_items["transaction_count"] / n_transactions
)

individual_items = (
    individual_items[
        individual_items["support"] >= 0.001
    ]
    .sort_values("support", ascending=False)
    .reset_index(drop=True)
)


In [29]:

# 3. Generate rules and itemsets with the number of itmes from 2 to 4
# ------------------------------------------------------------

results = list(
    apriori(
        transactions=transactions,
        min_support=0.001,
        min_confidence=0.10,
        min_lift=1.01,       # exploratory threshold
        min_length=2,
        max_length=4
    )
)


In [30]:

# 4. Extracting all the rules
# ------------------------------------------------------------

rule_records = []
itemset_records = []

for relation in results:
    itemset = tuple(sorted(relation.items))
    support = relation.support

    itemset_records.append({
        "itemset": ", ".join(itemset),
        "itemset_size": len(itemset),
        "support": support,
        "transaction_count": round(support * n_transactions)
    })

    for stat in relation.ordered_statistics:
        if not stat.items_base or not stat.items_add:
            continue

        rule_records.append({
            "left": ", ".join(sorted(stat.items_base)),
            "right": ", ".join(sorted(stat.items_add)),
            "itemset": ", ".join(itemset),
            "itemset_size": len(itemset),
            "support": support,
            "transaction_count": round(support * n_transactions),
            "confidence": stat.confidence,
            "lift": stat.lift
        })

itemsets_df = (
    pd.DataFrame(itemset_records)
    .drop_duplicates(subset="itemset")
)

itemsets_df


,itemset,itemset_size,support,transaction_count
0,"almonds, burgers",2,0.005199,39
1,"almonds, cake",2,0.003066,23
2,"almonds, chicken",2,0.002400,18
3,"almonds, chocolate",2,0.005999,45
4,"almonds, eggs",2,0.006532,49
...,...,...,...,...
5725,"mineral water, soup, spaghetti, turkey",4,0.001466,11
5726,"mineral water, spaghetti, tomatoes, turkey",4,0.001600,12
5727,"mineral water, spaghetti, tomatoes, whole whea...",4,0.001466,11
5728,"mineral water, spaghetti, turkey, whole wheat ...",4,0.001067,8


In [31]:

rules_df = pd.DataFrame(rule_records)
rules_df


,left,right,itemset,itemset_size,support,transaction_count,confidence,lift
0,almonds,burgers,"almonds, burgers",2,0.005199,39,0.254902,2.923577
1,almonds,cake,"almonds, cake",2,0.003066,23,0.150327,1.854607
2,almonds,chicken,"almonds, chicken",2,0.002400,18,0.117647,1.961046
3,almonds,chocolate,"almonds, chocolate",2,0.005999,45,0.294118,1.795099
4,almonds,eggs,"almonds, eggs",2,0.006532,49,0.320261,1.782108
...,...,...,...,...,...,...,...,...
15548,"pancakes, tomatoes","olive oil, spaghetti","olive oil, pancakes, spaghetti, tomatoes",4,0.001067,8,0.108108,4.714645
15549,"olive oil, pancakes, spaghetti",tomatoes,"olive oil, pancakes, spaghetti, tomatoes",4,0.001067,8,0.210526,3.078280
15550,"olive oil, pancakes, tomatoes",spaghetti,"olive oil, pancakes, spaghetti, tomatoes",4,0.001067,8,0.727273,4.177085
15551,"olive oil, spaghetti, tomatoes",pancakes,"olive oil, pancakes, spaghetti, tomatoes",4,0.001067,8,0.242424,2.550385


In [32]:

# Most frequent individual products
individual_items.head(10)


,item,transaction_count,support
0,mineral water,1788,0.238368
1,eggs,1348,0.179709
2,spaghetti,1306,0.174110
3,french fries,1282,0.170911
4,chocolate,1229,0.163845
5,green tea,991,0.132116
6,milk,972,0.129583
7,ground beef,737,0.098254
8,frozen vegetables,715,0.095321
9,pancakes,713,0.095054


The support value above shows the popularity of the individual items. for example, we can find mineral water in 23.8% of the transactions.

In [33]:

# Most popular itemsets
popular_itemsets = itemsets_df.sort_values(
    "support", ascending=False).head(10)
popular_itemsets


,itemset,itemset_size,support,transaction_count
1060,"mineral water, spaghetti",2,0.059725,448
392,"chocolate, mineral water",2,0.052660,395
539,"eggs, mineral water",2,0.050927,382
1010,"milk, mineral water",2,0.047994,360
875,"ground beef, mineral water",2,0.040928,307
409,"chocolate, spaghetti",2,0.039195,294
892,"ground beef, spaghetti",2,0.039195,294
551,"eggs, spaghetti",2,0.036528,274
519,"eggs, french fries",2,0.036395,273
746,"frozen vegetables, mineral water",2,0.035729,268


In [38]:
# The strong rules satisfying the complete project thresholds
strong_rules = rules_df[
    (rules_df["support"] >= 0.004) &
    (rules_df["confidence"] >= 0.20) &
    (rules_df["lift"] >= 4)
].copy()
strong_rules

,left,right,itemset,itemset_size,support,transaction_count,confidence,lift
360,light cream,chicken,"chicken, light cream",2,0.004533,34,0.290598,4.843951
625,pasta,escalope,"escalope, pasta",2,0.005866,44,0.372881,4.700812
1218,whole wheat pasta,olive oil,"olive oil, whole wheat pasta",2,0.007999,60,0.271493,4.122410
1250,pasta,shrimp,"pasta, shrimp",2,0.005066,38,0.322034,4.506672
5708,"eggs, ground beef",herb & pepper,"eggs, ground beef, herb & pepper",3,0.004133,31,0.206667,4.178455
8010,"herb & pepper, spaghetti",ground beef,"ground beef, herb & pepper, spaghetti",3,0.006399,48,0.393443,4.004360
14162,"frozen vegetables, ground beef","mineral water, spaghetti","frozen vegetables, ground beef, mineral water,...",4,0.004399,33,0.259843,4.350622


In [39]:

# Highest-confidence rules
strong_rules.sort_values("confidence", ascending=False).head(10)


,left,right,itemset,itemset_size,support,transaction_count,confidence,lift
8010,"herb & pepper, spaghetti",ground beef,"ground beef, herb & pepper, spaghetti",3,0.006399,48,0.393443,4.004360
625,pasta,escalope,"escalope, pasta",2,0.005866,44,0.372881,4.700812
1250,pasta,shrimp,"pasta, shrimp",2,0.005066,38,0.322034,4.506672
360,light cream,chicken,"chicken, light cream",2,0.004533,34,0.290598,4.843951
1218,whole wheat pasta,olive oil,"olive oil, whole wheat pasta",2,0.007999,60,0.271493,4.122410
14162,"frozen vegetables, ground beef","mineral water, spaghetti","frozen vegetables, ground beef, mineral water,...",4,0.004399,33,0.259843,4.350622
5708,"eggs, ground beef",herb & pepper,"eggs, ground beef, herb & pepper",3,0.004133,31,0.206667,4.178455


The highest value of confidence is for {herb & pepper, spaghetti}--> {ground beef} with the values 0.39. This tells us that 39 percent of the transactions with the herb & pepper, spaghetti have also ground beef in them.

In [40]:

# Highest-lift rules
strong_rules.sort_values("lift", ascending=False).head(10)

,left,right,itemset,itemset_size,support,transaction_count,confidence,lift
360,light cream,chicken,"chicken, light cream",2,0.004533,34,0.290598,4.843951
625,pasta,escalope,"escalope, pasta",2,0.005866,44,0.372881,4.700812
1250,pasta,shrimp,"pasta, shrimp",2,0.005066,38,0.322034,4.506672
14162,"frozen vegetables, ground beef","mineral water, spaghetti","frozen vegetables, ground beef, mineral water,...",4,0.004399,33,0.259843,4.350622
5708,"eggs, ground beef",herb & pepper,"eggs, ground beef, herb & pepper",3,0.004133,31,0.206667,4.178455
1218,whole wheat pasta,olive oil,"olive oil, whole wheat pasta",2,0.007999,60,0.271493,4.122410
8010,"herb & pepper, spaghetti",ground beef,"ground beef, herb & pepper, spaghetti",3,0.006399,48,0.393443,4.004360


The rule {light cream} → {chicken} has the highest lift, with a value of 4.84. This means that chicken appears in transactions containing light cream at 4.84 times its overall occurrence rate across all transactions.
lift (chicken | light cream ) = P (chicken | light cream) / P(chicken)
